# Figure S3 - mutation homoplasy and lineage reversion across all simulations

Reviewer 1, comment 1a bundles two objections to `antigen-prime`'s per-event random antigenic vectors: that the same substitution has no repeatable antigenic effect, and that the model is not time-reversible. These have **opposite** answers, so the figure reports them separately.

Unlike the previous version, which rested on the single `flu-final` build, every panel here is a distribution over **all** swept simulations, which is what removes the cherry-picking objection.

Inputs are the two small tables written by `scripts/sweep_mutation_homoplasy.py` (submitted via `scripts/submit_homoplasy_sweep.sh`, since the per-run Auspice trees exist only on the cluster). See `specs/mutation_homoplasy.md` for the method, and in particular for why the null must be *matched* to the statistic.

## Imports and parameters

In [ ]:
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

HERE = Path.cwd()
sys.path.append(str(HERE.parent / 'scripts'))

from plot_mutation_homoplasy import RC_PARAMS, build_across_run_figure

BATCH = '2026-07-04-reviewer-runs'
AGG_DIR = HERE.parent / 'results' / 'aggregated' / BATCH
CAND_PATH = HERE.parent / 'data' / BATCH / 'antigen-outputs' / 'candidate_runs.csv'
FIG_DIR = HERE.parent.parent / 'antigen-tex' / 'reviews' / 'round1' / 'figures'

# The unfiltered progeny threshold is the headline; the sweep also emits a
# filtered one as the control against transient origins.
MIN_ORIGIN_PROGENY = 1
FIG_PREFIX = 'figureS3_mutation_homoplasy'

## Load the sweep output and restrict to the flu-like candidate runs

The sweep covers every run that has a tree; filtering to the candidates happens here, using the same `cand_keys` idiom as figures S4 and S6.

In [ ]:
summary = pd.read_csv(AGG_DIR / 'mutation_homoplasy_by_run.csv')
similar = pd.read_csv(AGG_DIR / 'mutation_homoplasy_similar_background_by_k.csv')

cand = pd.read_csv(CAND_PATH)


def config_from_path(p):
    m = re.search(r'simulations/([^/]+)/run_', p)
    assert m is not None, f'cannot parse config from candidate path: {p!r}'
    return m.group(1)


cand['config'] = cand['path'].map(config_from_path)
cand_keys = set(zip(cand['config'], cand['run'].astype(int)))


def keep_candidates(df, label):
    keys = list(zip(df['config'], df['run'].astype(int)))
    kept = df[[k in cand_keys for k in keys]].copy()
    assert not kept.empty, f'no candidate runs survived the join for {label}'
    print(f'{label}: {len(kept)} rows from {kept.groupby(["config", "run"]).ngroups} candidate runs')
    return kept


summary = keep_candidates(summary, 'per-run summary')
similar = keep_candidates(similar, 'similar-background counts')
similar = similar[similar['min_origin_progeny'] == MIN_ORIGIN_PROGENY]

# Runs whose sequence tier failed carry tree-only statistics; report them rather
# than letting them silently shrink panels C and D.
partial = summary[summary['notes'].fillna('') != '']
print(f'runs with tree-only statistics: {len(partial)} of {len(summary)}')

## Figure S3

In [ ]:
with plt.rc_context(RC_PARAMS):
    fig = build_across_run_figure(summary, similar)
plt.show()

In [ ]:
FIG_DIR.mkdir(parents=True, exist_ok=True)
for suffix in ('pdf', 'png'):
    path = FIG_DIR / f'{FIG_PREFIX}.{suffix}'
    fig.savefig(path, dpi=300, bbox_inches='tight')
    print(f'Wrote {path}')

**Figure S3.** Mutation homoplasy and lineage reversion across all flu-like candidate `antigen-prime` simulations. Each point is one simulation.

**(A)** Fraction of distinct amino-acid substitutions that arise independently on two or more branches of the inferred tree, by site class. Recurrence is common, so the response does not claim otherwise.

**(B)** Fraction of substitutions participating in a gain-then-loss cycle along a single lineage, i.e. where the exact reverse substitution arises on a branch strictly below a forward origin. This is the quantity that speaks to time-reversibility, and unlike recurrence it is rare. Counts here are directed: a substitution and its reverse are two keys but one reversible pair.

**(C)** Fraction of recurrent substitutions whose independent origins arose within *k* amino acids of each other, against a matched null (grey dashed). Faint lines are individual simulations, bold lines the across-simulation median. Restricted to substitutions with exactly two independent origins, so there is a single pairwise distance and no minimum-selection effect; the matched null is therefore the distance between two arbitrarily chosen origin backgrounds.

**(D)** Median amino-acid distance between the backgrounds of a substitution's own independent origins versus the median distance between random pairs of origin backgrounds, one point per simulation. Points on the dashed line indicate that a substitution's own origins are no more similar than two arbitrary origins.

## Numbers quoted in the response letter

Pooled across candidate runs, weighting by counts rather than averaging per-run fractions (runs differ in size, and the parameter configs have unequal numbers of replicates).

In [ ]:
def pooled(numerator, denominator):
    return summary[numerator].sum() / summary[denominator].sum()


for site_class in ('epitope', 'non_epitope'):
    rec = pooled(f'n_recurrent_{site_class}', f'n_substitutions_{site_class}')
    rev = pooled(f'n_lineage_cycle_{site_class}', f'n_substitutions_{site_class}')
    print(f'{site_class:12s} recurrence {rec:6.1%}   lineage reversion {rev:6.1%}')

print()
pooled_similar = (
    similar[similar['statistic'] == 'two_origins']
    .groupby(['site_class', 'k'])[['n_within_k', 'n_total', 'n_null_within_k', 'n_null_total']]
    .sum()
)
pooled_similar['observed'] = pooled_similar['n_within_k'] / pooled_similar['n_total']
pooled_similar['matched_null'] = (
    pooled_similar['n_null_within_k'] / pooled_similar['n_null_total']
)
print('Similar-background recurrence, exactly-2-origin substitutions:')
print((100 * pooled_similar[['observed', 'matched_null']]).round(1).to_string())